# 02 · The noise library

Every noise source in `api.noise` is a process with independent, stationary increments, described by one function: its cumulant generating function per unit time,

$$K(\theta) = \lim_{\mathrm dt\to 0}\frac{1}{\mathrm dt}\log \mathbb E\, e^{\theta\,\mathrm d\eta}, \qquad \kappa_m = K^{(m)}(0)\ \text{(the $m$-th cumulant rate)} .$$

A source coupled to fields with coefficients $c_x$ adds $-K\big(\sum_x c_x \tilde x\big)$ to the action, so every cumulant of the noise becomes a vertex. Only infinitely divisible laws are offered (they are the ones that define a process). All moments must be finite: heavy-tailed stable laws are excluded.

This notebook lists the whole library — sources, jump laws for compound Poisson processes, and composites — evaluated live from the code, then works one example: a Lévy-driven Ornstein–Uhlenbeck process, whose stationary cumulants are known exactly.

Run time: about one minute.

In [1]:
%matplotlib inline
try:
    preparser(False)   # Sage kernel: plain Python syntax and numbers in the cells below
except NameError:
    pass
import os, sys, time, warnings
import numpy as np
import matplotlib.pyplot as plt
# locate the repository root (the folder holding the 'api' package) from any working directory
_root = os.path.abspath('')
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, 'api')):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)
os.chdir(_root)
def ells(r):   # C(tau) per loop order (real parts), a list indexed by the loop order
    return [np.real(np.asarray(r['C_tau_by_ell'][l])) for l in sorted(r['C_tau_by_ell'])]
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

In [2]:
import inspect
from IPython.display import Markdown, display
import api.noise as N
from sage.all import SR, latex

def sig(cls):
    return ', '.join(str(p) for p in inspect.signature(cls).parameters.values())

def k(src, m):
    return float(SR(src.cumulant(m)))

## 1. Sources

One row per source class. The example column is the instance used for the numbers; $K(\theta)$ is shown symbolically where the parameters can stay symbolic.

In [3]:
examples = {
    'Gaussian':                 (N.Gaussian(var='s2', mean='m'),            N.Gaussian(var=2.0, mean=0.1)),
    'Poisson':                  (N.Poisson(rate='r', size='a'),              N.Poisson(rate=2.0, size=0.5)),
    'CompoundPoisson':          (None,                                       N.CompoundPoisson(rate=2.0, jump=N.Exponential(mean=0.5))),
    'GammaProcess':             (N.GammaProcess(shape_rate='a', scale='b'),  N.GammaProcess(shape_rate=2.0, scale=0.5)),
    'InverseGaussianProcess':   (N.InverseGaussianProcess(mean_rate='m', shape='l'), N.InverseGaussianProcess(mean_rate=1.0, shape=2.0)),
    'VarianceGamma':            (N.VarianceGamma(mu='m', sigma='s', nu='n'), N.VarianceGamma(mu=0.1, sigma=0.3, nu=0.5)),
    'NormalInverseGaussian':    (None,                                       N.NormalInverseGaussian(alpha=2.0, beta=0.5, delta=1.0)),
    'CGMY':                     (None,                                       N.CGMY(C=1.0, G=2.0, M=3.0, Y=0.5)),
    'TemperedStableSubordinator': (None,                                     N.TemperedStableSubordinator(alpha=0.5, c=1.0, lam=2.0)),
    'Meixner':                  (None,                                       N.Meixner(alpha=1.0, beta=0.3, delta=1.0)),
    'BilateralGamma':           (N.BilateralGamma(alpha_p='ap', lam_p='lp', alpha_m='am', lam_m='lm'), N.BilateralGamma(alpha_p=1.0, lam_p=2.0, alpha_m=0.5, lam_m=1.0)),
    'Skellam':                  (N.Skellam(rate_plus='rp', rate_minus='rm'), N.Skellam(rate_plus=2.0, rate_minus=1.0)),
    'NegativeBinomialProcess':  (N.NegativeBinomialProcess(shape_rate='r', p='p'), N.NegativeBinomialProcess(shape_rate=1.0, p=0.3)),
    'Cumulants':                (None,                                       N.Cumulants(kappas=[0.1, 1.0, 0.2, 0.05])),
    'CGF':                      (N.CGF('lam*(exp(theta) - 1) + s2*theta^2/2'), N.CGF('2*(exp(theta) - 1) + theta^2/2')),
}
rows = ['| Source | Parameters | $K(\\theta)$ | example | $\\kappa_1$ | $\\kappa_2$ | $\\kappa_3$ | $\\kappa_4$ |',
        '|---|---|---|---|---|---|---|---|']
for name in N.SOURCE_TYPES:
    sym, ex = examples[name]
    K = f'${latex(sym.cgf())}$' if sym is not None else 'see `cgf()`'
    exs = ', '.join(f'{a}={v}' for a, v in ex.params.items()) if name not in ('CompoundPoisson', 'Cumulants', 'CGF') else \
          {'CompoundPoisson': 'rate=2, jump=Exponential(0.5)', 'Cumulants': 'kappas=[0.1, 1, 0.2, 0.05]',
           'CGF': '2(e^θ−1)+θ²/2'}[name]
    ks = ' | '.join(f'{k(ex, m):.4g}' for m in (1, 2, 3, 4))
    rows.append(f'| `{name}` | `{sig(getattr(N, name))}` | {K} | {exs} | {ks} |')
display(Markdown('\n'.join(rows)))

| Source | Parameters | $K(\theta)$ | example | $\kappa_1$ | $\kappa_2$ | $\kappa_3$ | $\kappa_4$ |
|---|---|---|---|---|---|---|---|
| `Gaussian` | `var, mean=0` | $\frac{1}{2} \, s_{2} \theta^{2} + m \theta$ | var=2.0, mean=0.1 | 0.1 | 2 | 0 | 0 |
| `Poisson` | `rate, size=1` | $r {\left(e^{\left(a \theta\right)} - 1\right)}$ | rate=2.0, size=0.5 | 1 | 0.5 | 0.25 | 0.125 |
| `CompoundPoisson` | `rate, jump: 'JumpDistribution'` | see `cgf()` | rate=2, jump=Exponential(0.5) | 1 | 1 | 1.5 | 3 |
| `GammaProcess` | `shape_rate, scale` | $-a \log\left(-b \theta + 1\right)$ | shape_rate=2.0, scale=0.5 | 1 | 0.5 | 0.5 | 0.75 |
| `InverseGaussianProcess` | `mean_rate, shape` | $-\frac{l {\left(\sqrt{-\frac{2 \, m^{2} \theta}{l} + 1} - 1\right)}}{m}$ | mean_rate=1.0, shape=2.0 | 1 | 0.5 | 0.75 | 1.875 |
| `Cumulants` | `kappas: 'Sequence', mean=0` | see `cgf()` | kappas=[0.1, 1, 0.2, 0.05] | 0 | 0.1 | 1 | 0.2 |
| `CGF` | `expression` | $\frac{1}{2} \, s_{2} \theta^{2} + \mathit{lam} {\left(e^{\theta} - 1\right)}$ | 2(e^θ−1)+θ²/2 | 2 | 3 | 2 | 2 |
| `VarianceGamma` | `mu, sigma, nu, drift=0` | $-\frac{\log\left(-\frac{1}{2} \, n s^{2} \theta^{2} - m n \theta + 1\right)}{n}$ | mu=0.1, sigma=0.3, nu=0.5, drift=0 | 0.1 | 0.095 | 0.014 | 0.01492 |
| `NormalInverseGaussian` | `alpha, beta, delta, mu=0` | see `cgf()` | alpha=2.0, beta=0.5, delta=1.0, mu=0 | 0.2582 | 0.5508 | 0.2203 | 0.5875 |
| `CGMY` | `C, G, M, Y` | see `cgf()` | C=1.0, G=2.0, M=3.0, Y=0.5 | -0.23 | 0.4839 | -0.1497 | 0.3648 |
| `TemperedStableSubordinator` | `alpha, c, lam` | see `cgf()` | alpha=0.5, c=1.0, lam=2.0 | 1.253 | 0.3133 | 0.235 | 0.2937 |
| `Meixner` | `alpha, beta, delta, mu=0` | see `cgf()` | alpha=1.0, beta=0.3, delta=1.0, mu=0 | 0.1511 | 0.5114 | 0.07729 | 0.2732 |
| `BilateralGamma` | `alpha_p, lam_p, alpha_m, lam_m` | $-\mathit{am} \log\left(\frac{\theta}{\mathit{lm}} + 1\right) - \mathit{ap} \log\left(-\frac{\theta}{\mathit{lp}} + 1\right)$ | alpha_p=1.0, lam_p=2.0, alpha_m=0.5, lam_m=1.0 | 0 | 0.75 | -0.75 | 3.375 |
| `Skellam` | `rate_plus, rate_minus` | $\mathit{rm} {\left(e^{\left(-\theta\right)} - 1\right)} + \mathit{rp} {\left(e^{\theta} - 1\right)}$ | rate_plus=2.0, rate_minus=1.0 | 1 | 3 | 1 | 3 |
| `NegativeBinomialProcess` | `shape_rate, p` | $-r {\left(\log\left(-p e^{\theta} + 1\right) - \log\left(-p + 1\right)\right)}$ | shape_rate=1.0, p=0.3 | 0.4286 | 0.6122 | 1.137 | 2.861 |

Notes on the sources:

* `CGMY`: $Y<0$ is a finite-activity compound Poisson process, $0<Y<1$ a difference of tempered-stable subordinators, $1<Y<2$ the compensated process; $Y = 0$ is the bilateral-gamma limit (use `BilateralGamma`) and $Y = 1$ is refused.
* `Cumulants(kappas)` declares the cumulant rates directly (a polynomial CGF); `CGF(expression)` takes any CGF in `theta`. Neither has a sampler.
* A parameter may depend on fields (a state-dependent rate or jump size): `Poisson(rate='g*(1 + x)', size=1)`.
* Sums: `a + b` of two sources is a `SumSource`, with $K = K_a + K_b$.

## 2. Jump laws for `CompoundPoisson(rate, jump)`

$K(\theta) = \lambda\,(M_J(\theta) - 1)$ with $M_J$ the moment generating function of the jump, so $\kappa_m = \lambda\,\mathbb E[J^m]$. The table lists the raw jump moments ($\lambda = 1$).

In [4]:
jumps = [N.Dirac(a=0.5), N.Exponential(mean=0.5), N.Gamma(shape=2.0, scale=0.5), N.GaussianJump(mu=0.2, sigma=0.5),
         N.Laplace(b=0.5), N.Uniform(lo=0.0, hi=1.0), N.Bernoulli(p=0.3), N.Binomial(n=4, p=0.3),
         N.FinitePMF(values=[-1, 1, 2], probs=[0.2, 0.5, 0.3]), N.Geometric(p=0.4), N.PoissonJump(mean=1.5),
         N.NegativeBinomial(r=2, p=0.3), N.Logarithmic(p=0.4), N.DoubleExponential(p=0.6, eta_up=3.0, eta_down=2.0),
         N.AsymmetricLaplace(scale_up=0.5, scale_down=0.3), N.Hyperexponential(probs=[0.3, 0.7], means=[0.2, 1.0]),
         N.PhaseType(alpha=[1, 0], T=[[-2, 2], [0, -3]]), N.ChiSquared(df=3), N.LogNormal(mu=0.0, sigma=0.3),
         N.Weibull(scale=1.0, shape=2.0)]
rows = ['| Jump law | Parameters | example | $E J$ | $E J^2$ | $E J^3$ | $E J^4$ |', '|---|---|---|---|---|---|---|']
for J in jumps:
    cp = N.CompoundPoisson(rate=1, jump=J)
    exs = ', '.join(f'{a}={v}' for a, v in J.params.items())
    rows.append(f'| `{type(J).__name__}` | `{sig(type(J))}` | {exs} | ' + ' | '.join(f'{k(cp, m):.4g}' for m in (1, 2, 3, 4)) + ' |')
display(Markdown('\n'.join(rows)))

| Jump law | Parameters | example | $E J$ | $E J^2$ | $E J^3$ | $E J^4$ |
|---|---|---|---|---|---|---|
| `Dirac` | `a` | a=0.5 | 0.5 | 0.25 | 0.125 | 0.0625 |
| `Exponential` | `mean` | mean=0.5 | 0.5 | 0.5 | 0.75 | 1.5 |
| `Gamma` | `shape, scale` | shape=2.0, scale=0.5 | 1 | 1.5 | 3 | 7.5 |
| `GaussianJump` | `mu, sigma` | mu=0.2, sigma=0.5 | 0.2 | 0.29 | 0.158 | 0.2491 |
| `Laplace` | `b` | b=0.5 | 0 | 0.5 | 0 | 1.5 |
| `Uniform` | `lo, hi` | lo=0.0, hi=1.0 | 0.5 | 0.3333 | 0.25 | 0.2 |
| `Bernoulli` | `p` | p=0.3 | 0.3 | 0.3 | 0.3 | 0.3 |
| `Binomial` | `n, p` | n=4, p=0.3 | 1.2 | 2.28 | 5.088 | 12.84 |
| `FinitePMF` | `values: 'Sequence', probs: 'Sequence'` | a0=-1, p0=0.2, a1=1, p1=0.5, a2=2, p2=0.3 | 0.9 | 1.9 | 2.7 | 5.5 |
| `Geometric` | `p` | p=0.4 | 2.5 | 10 | 58.75 | 460 |
| `PoissonJump` | `mean` | mean=1.5 | 1.5 | 3.75 | 11.62 | 42.56 |
| `NegativeBinomial` | `r, p` | r=2, p=0.3 | 0.8571 | 1.959 | 6.052 | 23.96 |
| `Logarithmic` | `p` | p=0.4 | 1.305 | 2.175 | 5.075 | 16.68 |
| `DoubleExponential` | `p, eta_up, eta_down` | p=0.6, eta_up=3.0, eta_down=2.0 | -5.551e-17 | 0.3333 | -0.1667 | 0.7778 |
| `AsymmetricLaplace` | `scale_up, scale_down, loc=0` | scale_up=0.5, scale_down=0.3, loc=0 | 0.2 | 0.38 | 0.408 | 1.01 |
| `Hyperexponential` | `probs: 'Sequence', means: 'Sequence'` | p0=0.3, m0=0.2, p1=0.7, m1=1.0 | 0.76 | 1.424 | 4.214 | 16.81 |
| `PhaseType` | `alpha: 'Sequence', T: 'Sequence'` | a0=1, a1=0, T0_0=-2, T0_1=2, T1_0=0, T1_1=-3 | 0.8333 | 1.056 | 1.806 | 3.907 |
| `ChiSquared` | `df` | shape=3/2, scale=2 | 3 | 15 | 105 | 945 |
| `LogNormal` | `mu, sigma, order=8` | mu=0.0, sigma=0.3 | 1.046 | 1.197 | 1.499 | 2.054 |
| `Weibull` | `scale, shape, order=8` | scale=1.0, shape=2.0 | 0.8862 | 1 | 1.329 | 2 |

`LogNormal` and `Weibull` have no analytic moment generating function: they are carried by their raw moments up to `order` (default 8), so the action holds a polynomial CGF of that order. Choose `order` at least as large as the highest cumulant order a computation uses ($k$ plus the loop corrections).

## 3. Composites

* `merton_jump_diffusion(drift, sigma, rate, jump_mean, jump_sd)`: Brownian motion plus compound Poisson with Gaussian jumps.
* `kou_jump_diffusion(drift, sigma, rate, p, eta_up, eta_down)`: Brownian motion plus double-exponential jumps.
* `levy_driven_ou(source, rate='mu')`: the Ornstein–Uhlenbeck process $\mathrm dx = -\mu x\,\mathrm dt + \mathrm dL$ driven by any source; returns an unbuilt `SDE`.

In [5]:
mj = N.merton_jump_diffusion(drift=0.0, sigma=0.5, rate=1.0, jump_mean=0.1, jump_sd=0.2)
kj = N.kou_jump_diffusion(drift=0.0, sigma=0.5, rate=1.0, p=0.6, eta_up=3.0, eta_down=2.0)
for name, s in (('Merton', mj), ('Kou', kj)):
    print(f"{name:7s} {type(s).__name__}: kappa_1..4 =", [round(k(s, m), 5) + 0.0 for m in (1, 2, 3, 4)])

Merton  SumSource: kappa_1..4 = [0.1, 0.3, 0.013, 0.0073]
Kou     SumSource: kappa_1..4 = [0.0, 0.58333, -0.16667, 0.77778]


## 4. Worked example: a Lévy-driven OU process

For $\mathrm dx = -\mu x\,\mathrm dt + \mathrm dL$ with $L$ a Lévy process of cumulant rates $\kappa_n$, the stationary cumulants of $x$ are exactly

$$\kappa_n(x) = \frac{\kappa_n(L)}{n\,\mu}.$$

The model is linear, so the $n$-point function at equal times is a *tree* diagram with one $n$-th order noise vertex. We check $n = 2, 3, 4$ for a variance-gamma driver, then compare with a simulation that draws the exact variance-gamma increments (`source.sample`).

In [6]:
from api import compute_cumulants
src = N.VarianceGamma(mu=0.3, sigma=0.5, nu=0.5)
mu = 1.0
model = N.levy_driven_ou(src, rate='mu', rate_default=mu).build()

def tree_equal_time(n):
    r = compute_cumulants(model, k=n, max_ell=0, external_fields=[('x', 1)] * n,
                          tau_grid=np.array([0.0, 1.0]), parallel=False, verbose=False)
    if n == 2:
        return complex(r['C_tau_by_ell'][0][0]).real
    return complex(r['total_C_by_ell'][0](*([0.0] * n))).real

for n in (2, 3, 4):
    print(f"n={n}: tree {tree_equal_time(n):.6f}   exact kappa_n/(n mu) {k(src, n) / (n * mu):.6f}")

n=2: tree 0.147500   exact kappa_n/(n mu) 0.147500
n=3: tree 0.042000   exact kappa_n/(n mu) 0.042000
n=4: tree 0.041831   exact kappa_n/(n mu) 0.041831


In [7]:
from api.noise_samplers import simulate_ou_paths, euler_ou_cumulant
from scipy.stats import kstat
rng = np.random.default_rng(0)
dt = 0.01
paths = simulate_ou_paths(src, mu, T=200.0, dt=dt, n_paths=1000, rng=rng)
xs = paths[:, ::50]                                          # every 0.5 time units, per path
groups = np.array_split(np.arange(xs.shape[0]), 20)          # 20 groups of independent paths -> standard errors
print(f"{'n':>2} {'simulated':>17} {'Euler-exact':>12} {'continuum':>10}")
for n in (2, 3, 4):
    g = np.array([kstat(xs[ix].ravel(), n) for ix in groups])
    print(f"{n:>2} {g.mean():>9.4f} ± {g.std(ddof=1) / np.sqrt(len(g)):.4f} "
          f"{euler_ou_cumulant(k(src, n), n, mu, dt):>12.4f} {k(src, n) / (n * mu):>10.4f}")

 n         simulated  Euler-exact  continuum
 2    0.1484 ± 0.0008       0.1482     0.1475
 3    0.0427 ± 0.0008       0.0424     0.0420
 4    0.0410 ± 0.0011       0.0425     0.0418


The simulated cumulants (± one standard error) are compared with the exact stationary cumulants of the Euler recursion the sampler integrates, which differ from the continuum values by $O(\mu\,\mathrm dt)$. Higher sample cumulants are much noisier: the fourth needs many more samples than the second for the same relative error.

More detail: [`docs/noise_library.md`](../../docs/noise_library.md) lists the samplers (exact or controlled approximations), telegraph and Markov-modulated noise, and what the perturbation theory cannot represent.